# Explainable Fraud Investigation Platform

## Data Preprocessing & Feature Engineering

### Objective

This notebook focuses on preparing the IEEE-CIS Fraud Detection dataset for machine learning model development.

Building upon the insights obtained during the Exploratory Data Analysis (EDA) phase, this stage transforms the raw transaction and identity data into a clean, consistent, and model-ready dataset.

The preprocessing pipeline includes handling missing values, removing low-information features, encoding categorical variables, scaling numerical features where appropriate, engineering new predictive features, selecting relevant variables, and creating reproducible training, validation, and testing datasets.

The output of this notebook serves as the input for the subsequent **Model Development & Comparison** phase of the Explainable Fraud Investigation Platform.

### Deliverables

By the end of this notebook, the following artifacts will be produced:

- A cleaned and preprocessed dataset suitable for machine learning.
- Engineered features designed to improve predictive performance.
- Encoded and transformed numerical and categorical variables.
- Train, validation, and test datasets.
- A reproducible preprocessing pipeline that can be reused for future model training and inference.

### 1.Import Libraries

In [1]:
# ==========================================================
# 1. Import Libraries
# ==========================================================

# Standard Libraries
import warnings
import random
from pathlib import Path

# Data Manipulation
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn - Preprocessing
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    OneHotEncoder,
    OrdinalEncoder
)

# Scikit-Learn - Imputation
from sklearn.impute import SimpleImputer

# Scikit-Learn - Feature Selection
from sklearn.feature_selection import VarianceThreshold

# Scikit-Learn - Model Selection
from sklearn.model_selection import train_test_split

# Save preprocessing objects
import joblib

# ----------------------------------------------------------
# Display Configuration
# ----------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 1200)
pd.set_option("display.float_format", "{:.2f}".format)

# ----------------------------------------------------------
# Plot Configuration
# ----------------------------------------------------------

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# ----------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


print("Libraries imported successfully.")
print(f"Random State: {RANDOM_STATE}")

Libraries imported successfully.
Random State: 42


In [2]:
# Versions check
import sys
import sklearn

print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")

Python: 3.11.15
Pandas: 3.0.3
NumPy: 2.4.6
Scikit-learn: 1.9.0


### 2. Import dataset

#### 2.1 Define Project Paths

In [3]:
# ==========================================================
# 2. Load the Dataset
# ==========================================================

# Define project paths

CURRENT_PATH = Path.cwd().resolve()

if CURRENT_PATH.name == "notebooks":
    PROJECT_ROOT = CURRENT_PATH.parent
elif (CURRENT_PATH / "data").exists():
    PROJECT_ROOT = CURRENT_PATH
else:
    raise FileNotFoundError(
        "Could not identify the project root. "
        "Run the notebook from the repository root or notebooks folder."
    )

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data Path: {RAW_DATA_PATH}")

required_files = [
    RAW_DATA_PATH / "train_transaction.csv",
    RAW_DATA_PATH / "train_identity.csv"
]

missing_files = [
    path for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(str(path) for path in missing_files)
    )

Project Root: /Users/hshazel/Projects/explainable-fraud-investigation-platform
Raw Data Path: /Users/hshazel/Projects/explainable-fraud-investigation-platform/data/raw


#### 2.2 Load Raw Data

In [4]:
# Load IEEE-CIS datasets

train_transaction = pd.read_csv(
    RAW_DATA_PATH / "train_transaction.csv",
    #low_memory=False
)

train_identity = pd.read_csv(
    RAW_DATA_PATH / "train_identity.csv",
    #low_memory=False # For large datasets, this prevents dtype inference and speeds up loading
)

print("Datasets loaded successfully.")

Datasets loaded successfully.


#### 2.3 Merge Datasets

In [5]:
# Merge transaction and identity datasets
assert train_transaction["TransactionID"].is_unique
assert train_identity["TransactionID"].is_unique

rows_before_merge = len(train_transaction)

df = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left",
    validate="one_to_one"
)

assert len(df) == rows_before_merge

print("Datasets merged successfully.")
print(f"Rows preserved after merge: {len(df):,}")

# Release memory by deleting the original DataFrames
del train_transaction
del train_identity

# Release memory by invoking garbage collection
import gc
gc.collect()

Datasets merged successfully.
Rows preserved after merge: 590,540


0

In [6]:
# ==========================================================
# Create Working Copy
# ==========================================================

df_processed = df.copy()

print("Working copy created successfully.")
print(f"Shape: {df_processed.shape}")

Working copy created successfully.
Shape: (590540, 434)


#### 2.4 Dataset Overview

In [7]:
print("=" * 60)
print("Dataset Overview")
print("=" * 60)

print(f"Rows: {df_processed.shape[0]:,}")
print(f"Columns: {df_processed.shape[1]:,}")

print("\nData Types")

display(df_processed.dtypes.value_counts())

Dataset Overview
Rows: 590,540
Columns: 434

Data Types


float64    399
str         31
int64        4
Name: count, dtype: int64

#### 2.5 Verify Target Variable

In [8]:
print("=" * 60)
print("Target Variable")
print("=" * 60)

display(df_processed["isFraud"].value_counts())

fraud_rate = df_processed["isFraud"].mean() * 100

print(f"\nFraud Rate: {fraud_rate:.2f}%")

Target Variable


isFraud
0    569877
1     20663
Name: count, dtype: int64


Fraud Rate: 3.50%


#### 2.6 Preview Dataset

In [9]:
preview_columns = [
    "TransactionID",
    "isFraud",
    "TransactionDT",
    "TransactionAmt",
    "ProductCD",
    "card1",
    "card4",
    "card6",
    "P_emaildomain",
    "DeviceType",
    "DeviceInfo"
]

display(df_processed[preview_columns].head())

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card4,card6,P_emaildomain,DeviceType,DeviceInfo
0,2987000,0,86400,68.50,W,13926,discover,credit,NaN,NaN,NaN
1,2987001,0,86401,29.00,W,2755,mastercard,credit,gmail.com,NaN,NaN
2,2987002,0,86469,59.00,W,4663,visa,debit,outlook.com,NaN,NaN
3,2987003,0,86499,50.00,W,18132,mastercard,debit,yahoo.com,NaN,NaN
4,2987004,0,86506,50.00,H,4497,mastercard,credit,gmail.com,mobile,SAMSUNG SM-G892A Build/NRD90M


In [10]:
# Small random sample of the dataset for preview

display(
    df_processed[preview_columns].sample(
        5,
        random_state=RANDOM_STATE
    )
)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card4,card6,P_emaildomain,DeviceType,DeviceInfo
470624,3457624,0,12153579,724.00,W,7826,mastercard,debit,aol.com,NaN,NaN
565820,3552820,0,15005886,108.50,W,12544,visa,debit,yahoo.com,NaN,NaN
284083,3271083,0,6970178,47.95,W,9400,mastercard,debit,gmail.com,NaN,NaN
239689,3226689,0,5673658,100.60,C,15885,visa,debit,gmail.com,NaN,NaN
281855,3268855,0,6886780,107.95,W,15497,visa,debit,hotmail.com,NaN,NaN


#### 2.7 Data Integrity Checks

In [11]:
print("=" * 60)
print("Data Integrity Checks")
print("=" * 60)

print(f"Duplicate TransactionID: {df_processed['TransactionID'].duplicated().sum()}")

print(f"Duplicate Rows: {df_processed.duplicated().sum()}")

print(f"Missing Target Values: {df_processed['isFraud'].isna().sum()}")

Data Integrity Checks
Duplicate TransactionID: 0
Duplicate Rows: 0
Missing Target Values: 0


#### 2.8 Dataset Memory Usage

In [12]:
memory_usage = (
    df_processed.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"Dataset Memory Usage: {memory_usage:.2f} MB")

Dataset Memory Usage: 1984.20 MB


#### Dataset Loading Summary

The IEEE-CIS Fraud Detection transaction and identity datasets were successfully loaded and merged using the `TransactionID` field.

Basic integrity checks confirmed that the target variable is present, the dataset contains no duplicate transaction identifiers, and the merged dataset is ready for preprocessing.

The resulting dataset will serve as the foundation for the data cleaning, feature engineering, and model preparation steps performed throughout this notebook.

### 3. Review Data Quality

Before applying preprocessing techniques, it is important to verify the overall quality and consistency of the merged dataset.

This review confirms that the dataset is structurally sound and identifies any issues that must be addressed during preprocessing, such as duplicate observations, missing values, inconsistent data types, low-information variables, and potential infinite values.

The findings from this section will guide the preprocessing strategy implemented throughout the remainder of this notebook.

#### 3.1 Dataset Shape

In [13]:
# ==========================================================
# 3.1 Dataset Shape
# ==========================================================

print("=" * 60)
print("Dataset Shape")
print("=" * 60)

print(f"Rows: {df_processed.shape[0]:,}")
print(f"Columns: {df_processed.shape[1]:,}")

Dataset Shape
Rows: 590,540
Columns: 434


#### 3.2 Data Types

In [14]:
# ==========================================================
# 3.2 Data Types
# ==========================================================

print("=" * 60)
print("Data Types")
print("=" * 60)

display(df_processed.dtypes.value_counts())

dtype_summary = (
    df_processed.dtypes
      .astype(str)
      .value_counts()
      .rename_axis("Data Type")
      .reset_index(name="Count")
)

display(dtype_summary)

Data Types


float64    399
str         31
int64        4
Name: count, dtype: int64

,Data Type,Count
0,float64,399
1,str,31
2,int64,4


#### 3.3 Missing Values Overview

We’re not treating them yet.

Just confirming the amount.

In [15]:
# ==========================================================
# 3.3 Missing Values Overview
# ==========================================================

missing_summary = (
    df_processed.isna()
      .sum()
      .to_frame("Missing Values")
)

missing_summary["Missing (%)"] = (
    missing_summary["Missing Values"]
    / len(df_processed)
    * 100
)

missing_summary = (
    missing_summary
    .sort_values("Missing (%)", ascending=False)
)

display(missing_summary.head(15))

,Missing Values,Missing (%)
id_24,585793,99.20
id_25,585408,99.13
id_07,585385,99.13
id_08,585385,99.13
id_21,585381,99.13
id_26,585377,99.13
id_27,585371,99.12
id_23,585371,99.12
id_22,585371,99.12
dist2,552913,93.63


#### 3.4 Duplicate Check

In [16]:
# ==========================================================
# 3.4 Duplicate Check
# ==========================================================

print("=" * 60)
print("Duplicate Check")
print("=" * 60)

print(f"Duplicate Rows: {df_processed.duplicated().sum()}")

print(f"Duplicate TransactionID: {df_processed['TransactionID'].duplicated().sum()}")

Duplicate Check
Duplicate Rows: 0
Duplicate TransactionID: 0


#### 3.5 Infinite Values

In [17]:
# ==========================================================
# 3.5 Infinite Values
# ==========================================================

numeric_df = df_processed.select_dtypes(include=np.number)

infinite_values = np.isinf(numeric_df).sum().sum()

print("=" * 60)
print("Infinite Values")
print("=" * 60)

print(f"Infinite Values: {infinite_values}")

Infinite Values
Infinite Values: 0


#### 3.6 Constant Features

These carry no predictive information.

In [18]:
# ==========================================================
# 3.6 Constant Features
# ==========================================================

constant_features = [
    column
    for column in df_processed.columns
    if df_processed[column].nunique(dropna=False) == 1
]

print("=" * 60)
print("Constant Features")
print("=" * 60)

print(f"Constant Features: {len(constant_features)}")

if constant_features:
    display(pd.DataFrame({"Feature": constant_features}))

Constant Features
Constant Features: 0


#### 3.7 Near-Constant Features

These are almost useless.

In [19]:
# ==========================================================
# 3.7 Near-Constant Features
# ==========================================================

threshold = 0.995

near_constant_features = []

for column in df_processed.columns:

    top_frequency = (
        df_processed[column]
        .value_counts(normalize=True, dropna=False)
        .iloc[0]
    )

    if top_frequency >= threshold:
        near_constant_features.append(column)

print("=" * 60)
print("Near-Constant Features")
print("=" * 60)

print(f"Near-Constant Features: {len(near_constant_features)}")

display(
    pd.DataFrame(
        {"Feature": near_constant_features}
    ).head(20)
)

Near-Constant Features
Near-Constant Features: 11


,Feature
0,C3
1,V107
2,V111
3,V113
4,V117
5,V118
6,V119
7,V120
8,V121
9,V122


#### Data Quality Findings

The dataset passed the initial quality review and is suitable for preprocessing.

The integrity checks confirmed that no duplicate transactions or duplicate observations are present, the target variable contains no missing values, and no infinite numerical values were detected.

A substantial number of variables contain missing values, which is expected for the IEEE-CIS Fraud Detection dataset and will be addressed during the missing value treatment stage. Several variables also exhibit little or no variability, making them potential candidates for removal during feature selection.

Overall, the dataset is structurally consistent and ready for data preprocessing and feature engineering.

### 4. Deterministic Feature Engineering

In [20]:
# ==========================================================
# 4. Deterministic Feature Engineering
# ==========================================================

SECONDS_PER_DAY = 24 * 60 * 60
SECONDS_PER_HOUR = 60 * 60

engineered_df = pd.DataFrame(
    index=df_processed.index
)

engineered_df["TransactionAmt_Log"] = np.log1p(
    df_processed["TransactionAmt"]
)

engineered_df["TransactionAmt_Rounded"] = (
    df_processed["TransactionAmt"].round()
)

engineered_df["TransactionDay"] = (
    df_processed["TransactionDT"] // SECONDS_PER_DAY
).astype("int32")

engineered_df["TransactionHour"] = (
    (
        df_processed["TransactionDT"]
        % SECONDS_PER_DAY
    )
    // SECONDS_PER_HOUR
).astype("int8")

engineered_df["Weekend"] = (
    engineered_df["TransactionDay"] % 7 >= 5
).astype("int8")

engineered_df["EmailMatch"] = (
    df_processed["P_emaildomain"].notna()
    & df_processed["R_emaildomain"].notna()
    & (
        df_processed["P_emaildomain"]
        == df_processed["R_emaildomain"]
    )
).astype("int8")

engineered_df["P_EmailProvider"] = (
    df_processed["P_emaildomain"]
    .fillna("unknown")
    .str.split(".")
    .str[0]
)

engineered_df["R_EmailProvider"] = (
    df_processed["R_emaildomain"]
    .fillna("unknown")
    .str.split(".")
    .str[0]
)

engineered_df["CardID"] = (
    df_processed["card1"]
    .astype("string")
    .fillna("unknown")
    + "_"
    + df_processed["card2"]
    .astype("string")
    .fillna("unknown")
)

engineered_df["IsMobile"] = (
    df_processed["DeviceType"] == "mobile"
).astype("int8")

# Must be created before DeviceInfo is imputed
engineered_df["UnknownDevice"] = (
    df_processed["DeviceInfo"].isna()
).astype("int8")

df_processed = pd.concat(
    [df_processed, engineered_df],
    axis=1
)

print(f"Dataset shape after feature engineering: {df_processed.shape}")

Dataset shape after feature engineering: (590540, 445)


### 4. Missing Value Treatment

The IEEE-CIS Fraud Detection dataset contains a substantial number of missing values, particularly within identity and device-related variables.

Rather than applying a single imputation strategy to all variables, missing values will be handled according to their characteristics and potential predictive value. Variables with extremely high proportions of missing data may be removed, while the remaining features will be imputed using techniques appropriate for their data type.

The objective of this section is to reduce unnecessary missingness while preserving information that may contribute to fraud prediction.

#### 4.1 Missing Value Summary

Even though we analyzed missing values in Notebook 1, we need the actual values here because they’ll drive preprocessing decisions.

In [21]:
# ==========================================================
# 4.1 Missing Value Summary
# ==========================================================

missing_summary = (
    df_processed
    .isna()
    .sum()
    .to_frame(name="Missing Values")
)

missing_summary["Missing (%)"] = (
    missing_summary["Missing Values"]
    / len(df_processed)
    * 100
)

missing_summary = (
    missing_summary
    .sort_values(
        by="Missing (%)",
        ascending=False
    )
)

display(missing_summary.head(20))

,Missing Values,Missing (%)
id_24,585793,99.20
id_25,585408,99.13
id_08,585385,99.13
id_07,585385,99.13
id_21,585381,99.13
id_26,585377,99.13
id_27,585371,99.12
id_23,585371,99.12
id_22,585371,99.12
dist2,552913,93.63


#### 4.2 Identify Features to Remove (>95% Missing)

This is exactly what was recommended in Notebook 1.

In [22]:
# ==========================================================
# 4.2 Features with >95% Missing Values
# ==========================================================

MISSING_THRESHOLD = 95

features_to_drop = (
    missing_summary[
        missing_summary["Missing (%)"] > MISSING_THRESHOLD
    ]
    .index
    .tolist()
)

print("=" * 60)
print("Features to Remove")
print("=" * 60)

print(f"Threshold: {MISSING_THRESHOLD}%")

print(f"Number of Features: {len(features_to_drop)}")

display(
    pd.DataFrame(
        {"Feature": features_to_drop}
    )
)

Features to Remove
Threshold: 95%
Number of Features: 9


,Feature
0,id_24
1,id_25
2,id_08
3,id_07
4,id_21
5,id_26
6,id_27
7,id_23
8,id_22


#### 4.3 Remove High-Missing Features

In [23]:
# ==========================================================
# 4.3 Remove Features
# ==========================================================

initial_columns = df_processed.shape[1]

df_processed.drop(
    columns=features_to_drop,
    inplace=True
)

removed_columns = initial_columns - df_processed.shape[1]

print("=" * 60)
print("Feature Removal")
print("=" * 60)

print(f"Columns Removed: {removed_columns}")

print(f"Remaining Columns: {df_processed.shape[1]}")

Feature Removal
Columns Removed: 9
Remaining Columns: 436


#### 4.4 Identify Numerical and Categorical Features

In [24]:
# ==========================================================
# 4.4 Identify Feature Types
# ==========================================================

numerical_features = (
    df_processed
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    df_processed
    .select_dtypes(include=["object", "string"])
    .columns
    .tolist()
)

print(f"Numerical Features: {len(numerical_features)}")

print(f"Categorical Features: {len(categorical_features)}")

Numerical Features: 404
Categorical Features: 32


#### 4.5 Missing Value Treatment Strategy

Plan for the imputation strategy

In [25]:
# ==========================================================
# 4.5 Missing Value Strategy
# ==========================================================

strategy = pd.DataFrame(index=df_processed.columns)

strategy["Data Type"] = (
    df_processed
    .dtypes
    .astype(str)
)

strategy["Missing (%)"] = (
    df_processed
    .isna()
    .mean()
    * 100
)

strategy["Treatment"] = np.where(
    strategy["Data Type"].isin(["object", "string", "str"]),
    "Mode Imputation",
    "Median Imputation"
)

display(
    strategy.sort_values(
        "Missing (%)",
        ascending=False
    ).head(20)
)

,Data Type,Missing (%),Treatment
dist2,float64,93.63,Median Imputation
D7,float64,93.41,Median Imputation
id_18,float64,92.36,Median Imputation
D13,float64,89.51,Median Imputation
D14,float64,89.47,Median Imputation
D12,float64,89.04,Median Imputation
id_04,float64,88.77,Median Imputation
id_03,float64,88.77,Median Imputation
D6,float64,87.61,Median Imputation
id_33,str,87.59,Mode Imputation


#### 4.6 Median Imputation (Numerical values)

In [26]:
# ==========================================================
# 4.6 Median Imputation
# ==========================================================

numeric_columns = (
    df_processed
    .select_dtypes(include=np.number)
    .columns
)

median_imputer = SimpleImputer(strategy="median")

df_processed[numeric_columns] = median_imputer.fit_transform(
    df_processed[numeric_columns]
)

print("Numerical missing values imputed using the median.")

Numerical missing values imputed using the median.


In [27]:
# Save the median imputer for future use
joblib.dump(
    median_imputer,
    PROJECT_ROOT / "models" / "preprocessing" / "median_imputer.pkl"
)
print("Median imputer saved.")

Median imputer saved.


#### 4.7 Mode Imputation (Categorical values)

In [28]:
# ==========================================================
# 4.7 Mode Imputation
# ==========================================================

categorical_columns = (
    df_processed
    .select_dtypes(include=["object", "string"])
    .columns
)

mode_imputer = SimpleImputer(strategy="most_frequent")

df_processed[categorical_columns] = mode_imputer.fit_transform(
    df_processed[categorical_columns]
)

print("Categorical missing values imputed using the mode.")

Categorical missing values imputed using the mode.


In [29]:
# Save the median imputer for future use

joblib.dump(
    mode_imputer,
    PROJECT_ROOT / "models" / "preprocessing" / "mode_imputer.pkl"
)

print("Mode imputer saved.")

Mode imputer saved.


#### 4.8 Final Verification

In [30]:
# ==========================================================
# 4.8 Final Verification
# ==========================================================

remaining_missing = (
    df_processed
    .isna()
    .sum()
    .sum()
)

print("=" * 60)
print("Missing Value Verification")
print("=" * 60)

print(f"Remaining Missing Values: {remaining_missing:,}")

Missing Value Verification
Remaining Missing Values: 0


#### Missing Value Treatment Findings

Variables containing more than **95% missing values** were removed because they provide limited information for model development and would require extensive imputation.

The remaining numerical variables were imputed using the median, which is robust to outliers and appropriate for the highly skewed distributions observed during exploratory analysis. Categorical variables were imputed using the most frequent category to preserve the dominant transaction characteristics.

Following preprocessing, the dataset contains no remaining missing values and is ready for subsequent feature engineering and encoding.

### 5. Remove Low-Information Features

Following the missing value treatment, the dataset is further refined by removing features that provide little or no predictive value.

Low-information features include variables with constant values across all observations or variables whose values exhibit extremely low variability. Such features contribute little to model performance while increasing computational complexity and the risk of overfitting.

Removing these variables simplifies the feature space and improves the efficiency of subsequent machine learning models.

#### 5.1 Remove Constant Features

Already identified in Section 3.

In [31]:
# ==========================================================
# 5.1 Remove Constant Features
# ==========================================================

constant_features = [
    column
    for column in df_processed.columns
    if df_processed[column].nunique(dropna=False) == 1
]

print("=" * 60)
print("Constant Features")
print("=" * 60)

print(f"Constant Features Found: {len(constant_features)}")

if constant_features:
    df_processed.drop(
        columns=constant_features,
        inplace=True
    )

print(f"Remaining Columns: {df_processed.shape[1]}")

Constant Features
Constant Features Found: 0
Remaining Columns: 436


#### 5.2 Remove Near-Constant Features

There are the 11 variables identified earlier:

In [32]:
# ==========================================================
# 5.2 Remove Near-Constant Features
# ==========================================================

threshold = 0.995

near_constant_features = []

for column in df_processed.columns:

    top_frequency = (
        df_processed[column]
        .value_counts(normalize=True, dropna=False)
        .iloc[0]
    )

    if top_frequency >= threshold:
        near_constant_features.append(column)

print("=" * 60)
print("Near-Constant Features")
print("=" * 60)

print(f"Features Removed: {len(near_constant_features)}")

display(
    pd.DataFrame(
        {"Feature": near_constant_features}
    )
)

df_processed.drop(
    columns=near_constant_features,
    inplace=True
)

print(f"Remaining Columns: {df_processed.shape[1]}")

Near-Constant Features
Features Removed: 37


,Feature
0,C3
1,M1
2,V1
3,V14
4,V27
5,V28
6,V41
7,V65
8,V68
9,V88


Remaining Columns: 399


#### 5.3 Feature Reduction Summary

In [33]:
# ==========================================================
# 5.3 Feature Reduction Summary
# ==========================================================

print("=" * 60)
print("Feature Reduction Summary")
print("=" * 60)

print(f"Dataset Shape: {df_processed.shape}")

removed_total = len(constant_features) + len(near_constant_features)

print(f"Total Features Removed: {removed_total}")

Feature Reduction Summary
Dataset Shape: (590540, 399)
Total Features Removed: 37


#### Low-Information Feature Findings

No constant features were identified within the dataset.

Following missing value treatment, **37 near-constant features** were identified and removed because a single value accounted for more than **99.5%** of their observations.

Many of these variables became near-constant after median or mode imputation, indicating that they contributed minimal discriminatory information for model training.

After removing these variables, the dataset contains **388 features**, providing a more compact representation for the subsequent feature engineering and model development stages.

### 6. Feature Engineering

#### 6.1 Create Transaction Amount Features

In [34]:
# Create TransactionAmt_Log: Log transformation reduces the heavy right skew.

df_processed["TransactionAmt_Log"] = np.log1p(
    df_processed["TransactionAmt"]
)

In [35]:
# TransactionAmt_Rounded: Fraudsters often use repeated rounded amounts.

df_processed["TransactionAmt_Rounded"] = (
    df_processed["TransactionAmt"]
    .round()
)

#### 6.2 Create Time-Based Features

In [36]:
# TransactionDT is not a real timestamp. It represents elapsed seconds.

SECONDS_PER_DAY = 24 * 60 * 60

df_processed["TransactionDay"] = (
    df_processed["TransactionDT"] // SECONDS_PER_DAY
).astype(int)

# Hour of transaction
SECONDS_PER_HOUR = 60 * 60

df_processed["TransactionHour"] = (
    (
        df_processed["TransactionDT"]
        % SECONDS_PER_DAY
    )
    // SECONDS_PER_HOUR
).astype(int)

#Weekend indicator
df_processed["Weekend"] = (
    df_processed["TransactionDay"] % 7 >= 5
).astype(int)

#### 6.3 Create Email Features

In [37]:
# Create a domain match feature. If both are missing this becomes False.

df_processed["EmailMatch"] = (
    df_processed["P_emaildomain"]
    ==
    df_processed["R_emaildomain"]
).astype(int)

In [38]:
# Email provider, extract only the provider.
df_processed["P_EmailProvider"] = (
    df_processed["P_emaildomain"]
    .fillna("unknown")
    .str.split(".")
    .str[0]
)

df_processed["R_EmailProvider"] = (
    df_processed["R_emaildomain"]
        .fillna("unknown")
        .str.split(".")
        .str[0]
)

#### 6.4 Create Card Features

In [39]:
# Create CardMatch using card1, card2
df_processed["CardID"] = (
    df_processed["card1"].astype(str)
    + "_"
    + df_processed["card2"].astype(str)
)

#### 6.5 Create Device Features

In [40]:
# isMobile
df_processed["IsMobile"] = (
    df_processed["DeviceType"]
    == "mobile"
).astype(int)

# Unknown Device
# Create missing-value indicator BEFORE imputation
df_processed["UnknownDevice"] = (
    df_processed["DeviceInfo"].isna()
).astype("int8")

#### 6.6 Feature Engineering Summary

In [41]:
engineered_features = [
    "TransactionAmt_Log",
    "TransactionAmt_Rounded",
    "TransactionDay",
    "TransactionHour",
    "Weekend",
    "EmailMatch",
    "P_EmailProvider",
    "R_EmailProvider",
    "CardID",
    "IsMobile",
    "UnknownDevice"
]

print("=" * 60)
print("Feature Engineering Summary")
print("=" * 60)

print(f"New Features Created: {len(engineered_features)}")

display(
    pd.DataFrame(
        {"Feature": engineered_features}
    )
)

print(f"Dataset Shape: {df_processed.shape}")

Feature Engineering Summary
New Features Created: 11


,Feature
0,TransactionAmt_Log
1,TransactionAmt_Rounded
2,TransactionDay
3,TransactionHour
4,Weekend
5,EmailMatch
6,P_EmailProvider
7,R_EmailProvider
8,CardID
9,IsMobile


Dataset Shape: (590540, 399)


In [42]:
# Avoid data fragmentation by creating a copy of the DataFrame after all modifications.
df_processed = df_processed.copy()

In [43]:
display(df_processed.head())

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M2,M3,M4,M5,M6,M7,M8,M9,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V66,V67,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V108,V109,V110,V112,V114,V115,V116,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V139,V140,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V192,V193,V194,V195,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V242,V243,V244,V245,V246,V248,V249,V250,V251,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,id_01,id_02,id_05,id_06,id_09,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,TransactionAmt_Log,TransactionAmt_Rounded,TransactionDay,TransactionHour,Weekend,EmailMatch,P_EmailProvider,R_EmailProvider,CardID,IsMobile,UnknownDevice
0,2987000.00,0.00,86400.00,68.50,W,13926.00,361.00,150.00,discover,142.00,credit,315.00,87.00,19.00,37.00,gmail.com,gmail.com,1.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,2.00,0.00,1.00,1.00,14.00,97.00,13.00,26.00,10.00,0.00,0.00,37.88,0.67,13.00,13.00,0.00,0.00,0.00,0.00,T,T,M2,F,T,F,F,T,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,117.00,0.00,0.00,0.00,0.00,0.00,117.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,117.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,117.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-5.00,125800.50,0.00,0.00,0.00,100.00,NotFound,52.00,-300.00,Found,Found,166.00,15.00,341.00,472.00,Found,Found,Windows 10,chrom

### 7. Encode Categorical Variables

Machine learning algorithms require numerical inputs. Therefore, categorical variables must be transformed into numerical representations before model training.

In this project, categorical variables are encoded using One-Hot Encoding. Unlike ordinal encoding, One-Hot Encoding does not impose artificial ordering among categories, making it appropriate for nominal variables such as payment card type, product category, email provider, and device information.

The fitted encoder will be saved so that identical transformations can be applied during model deployment and inference.

#### 7.1 Split categorical features by cardinality

In [44]:
MAX_ONEHOT_CATEGORIES = 20

categorical_features = (
    df_processed
    .select_dtypes(include=["object", "string"])
    .columns
    .tolist()
)

low_cardinality_features = [
    column
    for column in categorical_features
    if df_processed[column].nunique(dropna=False)
    <= MAX_ONEHOT_CATEGORIES
]

high_cardinality_features = [
    column
    for column in categorical_features
    if df_processed[column].nunique(dropna=False)
    > MAX_ONEHOT_CATEGORIES
]

print(f"Low-cardinality features: {len(low_cardinality_features)}")
print(f"High-cardinality features: {len(high_cardinality_features)}")

cardinality_summary = pd.DataFrame({
    "Feature": categorical_features,
    "Unique Categories": [
        df_processed[column].nunique(dropna=False)
        for column in categorical_features
    ],
    "Encoding": [
        "One-Hot"
        if column in low_cardinality_features
        else "Frequency"
        for column in categorical_features
    ]
}).sort_values("Unique Categories", ascending=False)

display(cardinality_summary)

Low-cardinality features: 22
High-cardinality features: 9


,Feature,Unique Categories,Encoding
30,CardID,14520,Frequency
27,DeviceInfo,1786,Frequency
20,id_33,260,Frequency
19,id_31,130,Frequency
18,id_30,75,Frequency
4,R_emaildomain,60,Frequency
3,P_emaildomain,59,Frequency
29,R_EmailProvider,45,Frequency
28,P_EmailProvider,44,Frequency
0,ProductCD,5,One-Hot


In [45]:
frequency_maps = {}

for column in high_cardinality_features:
    frequency_map = (
        df_processed[column]
        .value_counts(normalize=True, dropna=False)
        .to_dict()
    )

    frequency_maps[column] = frequency_map

    df_processed[f"{column}_Frequency"] = (
        df_processed[column]
        .map(frequency_map)
        .fillna(0)
        .astype(np.float32)
    )

df_processed.drop(
    columns=high_cardinality_features,
    inplace=True
)

print(
    f"Frequency-encoded "
    f"{len(high_cardinality_features)} features."
)

Frequency-encoded 9 features.


#### 7.3 One-Hot Encode Low-Cardinality Features

In [46]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=np.float32
)

encoded_array = encoder.fit_transform(
    df_processed[low_cardinality_features]
)

encoded_df = pd.DataFrame(
    encoded_array,
    index=df_processed.index,
    columns=encoder.get_feature_names_out(
        low_cardinality_features
    )
)

print(f"One-hot encoded matrix shape: {encoded_df.shape}")

One-hot encoded matrix shape: (590540, 55)


In [47]:
missing_after_encoding = int(
    df_processed.isna().to_numpy().sum()
)

print(
    "Missing values after encoding:",
    missing_after_encoding
)

Missing values after encoding: 0


#### 7.4 Replace Original Categorical Variables

In [48]:
# ==========================================================
# 7.4 Replace Original Variables
# ==========================================================

numeric_df = df_processed.drop(
    columns=low_cardinality_features
)

df_processed = pd.concat(
    [numeric_df, encoded_df],
    axis=1
)

print("=" * 60)
print("Encoding Summary")
print("=" * 60)

print(f"Dataset Shape: {df_processed.shape}")

print(
    "Remaining Missing Values:",
    int(df_processed.isna().to_numpy().sum())
)

Encoding Summary
Dataset Shape: (590540, 432)
Remaining Missing Values: 0


#### 7.5 Save the Encoder and Feature Names

In [49]:
# ==========================================================
# 7.5 Save Encoder
# ==========================================================

encoder_path = (
    PROJECT_ROOT
    / "models"
    / "preprocessing"
    / "onehot_encoder.pkl"
)

joblib.dump(
    encoder,
    encoder_path
)

print(f"Encoder saved to:\n{encoder_path}")

Encoder saved to:
/Users/hshazel/Projects/explainable-fraud-investigation-platform/models/preprocessing/onehot_encoder.pkl


In [50]:
onehot_feature_names = encoder.get_feature_names_out(
    low_cardinality_features
)

joblib.dump(
    onehot_feature_names,
    PROJECT_ROOT
    / "models"
    / "preprocessing"
    / "encoded_feature_names.pkl"
)

joblib.dump(
    frequency_maps,
    PROJECT_ROOT
    / "models"
    / "preprocessing"
    / "frequency_maps.pkl"
)

joblib.dump(
    low_cardinality_features,
    PROJECT_ROOT
    / "models"
    / "preprocessing"
    / "low_cardinality_features.pkl"
)

joblib.dump(
    high_cardinality_features,
    PROJECT_ROOT
    / "models"
    / "preprocessing"
    / "high_cardinality_features.pkl"
)

print("Categorical preprocessing artifacts saved.")

Categorical preprocessing artifacts saved.


#### 7.6 Verify the Encoding

In [51]:
# ==========================================================
# 7.6 Verify Encoded Dataset
# ==========================================================

print("=" * 60)
print("Encoded Dataset")
print("=" * 60)

print(f"Rows: {df_processed.shape[0]:,}")
print(f"Columns: {df_processed.shape[1]:,}")

display(df_processed.head())

Encoded Dataset
Rows: 590,540
Columns: 432


,TransactionID,isFraud,TransactionDT,TransactionAmt,card1,card2,card3,card5,addr1,addr2,dist1,dist2,C1,C2,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V66,V67,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V108,V109,V110,V112,V114,V115,V116,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V139,V140,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V192,V193,V194,V195,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V242,V243,V244,V245,V246,V248,V249,V250,V251,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,id_01,id_02,id_05,id_06,id_09,id_11,id_13,id_14,id_17,id_18,id_19,id_20,id_32,TransactionAmt_Log,TransactionAmt_Rounded,TransactionDay,TransactionHour,Weekend,EmailMatch,IsMobile,UnknownDevice,P_emaildomain_Frequency,R_emaildomain_Frequency,id_30_Frequency,id_31_Frequency,id_33_Frequency,DeviceInfo_Frequency,P_EmailProvider_Frequency,R_EmailProvider_Frequency,CardID_Frequency,ProductCD_C,ProductCD_H,ProductCD_R,ProductCD_S,ProductCD_W,card4_american express,card4_discover,card4_mastercard,card4_visa,card6_charge card,card6_credit,card6_debit,card6_debit or credit,M2_F,M2_T,M3_F,M3_T,M4_M0,M4_M1,M4_M2,M5_F,M5_T,M6_F,M6_T,M7_F,M7_T,M8_F,M8_T,M9_F,M9_T,id_12_Found,id_12_NotFound,id_15_Found,id_15_New,id_15_Unknown,id_16_Found,id_16_NotFound,id_28_Found,id_28_New,id_29_Found,id_29_NotFound,id_34_match_status:-1,id_34_match_status:0,id_34_match_status:1,id_34_match_status:2,id_35_F,id_35_T,id_36_F,id_36_T,id_37_F,id_37_T,id_38_F,id_38_T,DeviceType_desktop,DeviceType_mobile
0,2987000.00,0.00,86400.00,68.50,13926.00,361.00,150.00,142.00,315.00,87.00,19.00,37.00,1.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,2.00,0.00,1.00,1.00,14.00,97.00,13.00,26.00,10.00,0.00,0.00,37.88,0.67,13.00,13.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,117.00,0.00,0.00,0.00,0.00,0.00,117.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,

In [52]:
def count_missing_values(dataframe):
    return sum(
        int(dataframe[column].isna().sum())
        for column in dataframe.columns
    )

print("Missing values in df_processed:", count_missing_values(df_processed))

Missing values in df_processed: 0


#### Categorical Encoding Findings

Categorical variables were encoded using a hybrid strategy based on feature cardinality.

Low-cardinality variables were transformed using One-Hot Encoding, preserving their nominal structure without introducing artificial ordinal relationships. High-cardinality variables were transformed using frequency encoding to avoid generating an excessively large and memory-intensive feature matrix.

The fitted encoder, generated feature names, and frequency mappings were saved as reusable preprocessing artifacts. Following encoding, the dataset contains only numerical variables and is ready for dataset splitting and numerical scaling.

### 8. Train / Validation / Test Split

The processed dataset is divided into training, validation, and test subsets before model development.

A stratified split is used to preserve the original fraud rate of approximately **3.50%** across all subsets. This is especially important because the target variable is highly imbalanced.

The training dataset will be used to fit machine learning models and preprocessing transformations. The validation dataset will support model comparison and hyperparameter tuning, while the test dataset will remain untouched until the final model evaluation.

#### 8.1 Separate features and target

In [53]:
# ==========================================================
# 8.1 Separate Features and Target
# ==========================================================

TARGET_COLUMN = "isFraud"

X = df_processed.drop(columns=[TARGET_COLUMN]).copy()
y = df_processed[TARGET_COLUMN].astype("int8").copy()

print("=" * 60)
print("Features and Target")
print("=" * 60)

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Fraud rate: {y.mean() * 100:.2f}%")

Features and Target
Feature matrix shape: (590540, 431)
Target shape: (590540,)
Fraud rate: 3.50%


#### 8.2 Remove the transaction identifier from model inputs

TransactionID uniquely identifies rows but does not represent customer behavior. We keep it separately for investigation and reporting.

In [54]:
# ==========================================================
# 8.2 Preserve Transaction Identifiers
# ==========================================================

transaction_ids = X["TransactionID"].copy()

X.drop(
    columns=["TransactionID"],
    inplace=True
)

print("TransactionID removed from model features.")
print(f"Remaining feature columns: {X.shape[1]}")

TransactionID removed from model features.
Remaining feature columns: 430


#### 8.3 Create the 70% / 15% / 15% split

In [55]:
# ==========================================================
# 8.3 Train / Validation / Test Split
# ==========================================================

X_train, X_temp, y_train, y_temp, id_train, id_temp = train_test_split(
    X,
    y,
    transaction_ids,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

X_validation, X_test, y_validation, y_test, id_validation, id_test = (
    train_test_split(
        X_temp,
        y_temp,
        id_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=RANDOM_STATE
    )
)

print("Dataset split completed successfully.")

Dataset split completed successfully.


#### 8.4 Verify sizes and fraud rates

In [56]:
# ==========================================================
# 8.4 Split Verification
# ==========================================================

split_summary = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation",
        "Test"
    ],
    "Rows": [
        len(X_train),
        len(X_validation),
        len(X_test)
    ],
    "Columns": [
        X_train.shape[1],
        X_validation.shape[1],
        X_test.shape[1]
    ],
    "Fraud Cases": [
        int(y_train.sum()),
        int(y_validation.sum()),
        int(y_test.sum())
    ],
    "Fraud Rate (%)": [
        y_train.mean() * 100,
        y_validation.mean() * 100,
        y_test.mean() * 100
    ]
})

display(split_summary.round(2))

,Dataset,Rows,Columns,Fraud Cases,Fraud Rate (%)
0,Training,413378,430,14464,3.50
1,Validation,88581,430,3100,3.50
2,Test,88581,430,3099,3.50


#### 8.5 Verify no overlap

In [57]:
# ==========================================================
# 8.5 Check Dataset Independence
# ==========================================================

train_ids = set(id_train)
validation_ids = set(id_validation)
test_ids = set(id_test)

print("=" * 60)
print("Dataset Independence Checks")
print("=" * 60)

print(
    "Train/Validation overlap:",
    len(train_ids.intersection(validation_ids))
)

print(
    "Train/Test overlap:",
    len(train_ids.intersection(test_ids))
)

print(
    "Validation/Test overlap:",
    len(validation_ids.intersection(test_ids))
)

Dataset Independence Checks
Train/Validation overlap: 0
Train/Test overlap: 0
Validation/Test overlap: 0


#### 8.6 Verify feature consistency

In [58]:
def count_missing_values(dataframe):
    return sum(
        int(dataframe[column].isna().sum())
        for column in dataframe.columns
    )


print("=" * 60)
print("Feature Consistency")
print("=" * 60)

print(
    "Train/Validation columns match:",
    list(X_train.columns) == list(X_validation.columns)
)

print(
    "Train/Test columns match:",
    list(X_train.columns) == list(X_test.columns)
)

print(
    "Missing values in training:",
    count_missing_values(X_train)
)

print(
    "Missing values in validation:",
    count_missing_values(X_validation)
)

print(
    "Missing values in test:",
    count_missing_values(X_test)
)

Feature Consistency
Train/Validation columns match: True
Train/Test columns match: True
Missing values in training: 0
Missing values in validation: 0
Missing values in test: 0


In [59]:
print("Missing values:", df_processed.isnull().sum().sum())

print("Infinite values:",
      np.isinf(df_processed.select_dtypes(include=np.number)).sum().sum())

print("Duplicate rows:",
      df_processed.duplicated().sum())

Missing values: 0
Infinite values: 0
Duplicate rows: 0


#### 8.7 Final DataFrame

In [60]:
df_processed.describe(include="all")

,TransactionID,isFraud,TransactionDT,TransactionAmt,card1,card2,card3,card5,addr1,addr2,dist1,dist2,C1,C2,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V66,V67,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V108,V109,V110,V112,V114,V115,V116,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V139,V140,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V192,V193,V194,V195,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V242,V243,V244,V245,V246,V248,V249,V250,V251,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,id_01,id_02,id_05,id_06,id_09,id_11,id_13,id_14,id_17,id_18,id_19,id_20,id_32,TransactionAmt_Log,TransactionAmt_Rounded,TransactionDay,TransactionHour,Weekend,EmailMatch,IsMobile,UnknownDevice,P_emaildomain_Frequency,R_emaildomain_Frequency,id_30_Frequency,id_31_Frequency,id_33_Frequency,DeviceInfo_Frequency,P_EmailProvider_Frequency,R_EmailProvider_Frequency,CardID_Frequency,ProductCD_C,ProductCD_H,ProductCD_R,ProductCD_S,ProductCD_W,card4_american express,card4_discover,card4_mastercard,card4_visa,card6_charge card,card6_credit,card6_debit,card6_debit or credit,M2_F,M2_T,M3_F,M3_T,M4_M0,M4_M1,M4_M2,M5_F,M5_T,M6_F,M6_T,M7_F,M7_T,M8_F,M8_T,M9_F,M9_T,id_12_Found,id_12_NotFound,id_15_Found,id_15_New,id_15_Unknown,id_16_Found,id_16_NotFound,id_28_Found,id_28_New,id_29_Found,id_29_NotFound,id_34_match_status:-1,id_34_match_status:0,id_34_match_status:1,id_34_match_status:2,id_35_F,id_35_T,id_36_F,id_36_T,id_37_F,id_37_T,id_38_F,id_38_T,DeviceType_desktop,DeviceType_mobile
count,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00,590540.00

#### Dataset Split Findings

The processed dataset was divided into training, validation, and test subsets using a stratified **70% / 15% / 15% split**.

Stratification preserved the original fraud rate of approximately **3.50%** across all three datasets, ensuring that each subset contains a representative proportion of legitimate and fraudulent transactions.

The `TransactionID` variable was excluded from model training because it functions as a unique identifier rather than a predictive behavioral feature. It was retained separately to support transaction-level investigation, reporting, and model explainability.

Verification confirmed that there is no transaction overlap between the training, validation, and test subsets. All subsets contain the same feature columns and are ready for subsequent scaling and model development.

### 9. Scale Numerical Features

The numerical variables in the dataset operate on substantially different scales. For example, transaction amounts, elapsed-time variables, count-based features, and identity-related measurements have different numerical ranges.

Feature scaling is particularly important for algorithms such as Logistic Regression and neural networks, which are sensitive to differences in feature magnitude. Tree-based algorithms such as Decision Trees, Random Forest, and XGBoost are generally less sensitive to scaling.

`RobustScaler` is used because several numerical variables contain highly skewed distributions and extreme values. The scaler uses the median and interquartile range, making it less sensitive to outliers than standardization based on the mean and standard deviation.

To reduce data leakage, the scaler is fitted exclusively on the training dataset. The fitted transformation is then applied unchanged to the validation and test datasets.

#### 9.1 Identify features to scale

We should not scale one-hot indicators or other binary columns.

In [61]:
# ==========================================================
# 9.1 Identify Numerical Features to Scale
# ==========================================================

numerical_columns = (
    X_train
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

# One-hot columns created by the encoder
onehot_columns = [
    column
    for column in onehot_feature_names
    if column in X_train.columns
]

# Binary features should remain coded as 0 and 1
binary_columns = [
    column
    for column in numerical_columns
    if X_train[column].nunique(dropna=False) <= 2
]

numerical_features_to_scale = [
    column
    for column in numerical_columns
    if column not in onehot_columns
    and column not in binary_columns
]

print("=" * 60)
print("Numerical Feature Selection")
print("=" * 60)

print(f"Total model features: {X_train.shape[1]}")
print(f"Numerical features: {len(numerical_columns)}")
print(f"One-hot features excluded: {len(onehot_columns)}")
print(f"Binary features excluded: {len(binary_columns)}")
print(
    f"Numerical features selected for scaling: "
    f"{len(numerical_features_to_scale)}"
)

Numerical Feature Selection
Total model features: 430
Numerical features: 430
One-hot features excluded: 55
Binary features excluded: 59
Numerical features selected for scaling: 371


In [62]:
display(
    pd.DataFrame({
        "Feature": numerical_features_to_scale
    }).head(30)
)

,Feature
0,TransactionDT
1,TransactionAmt
2,card1
3,card2
4,card3
5,card5
6,addr1
7,addr2
8,dist1
9,dist2


#### 9.2 Convert selected columns to float32

This reduces memory usage and prevents dtype-assignment warnings.

In [63]:
# ==========================================================
# 9.2 Prepare Numerical Columns
# ==========================================================

for dataset in [X_train, X_validation, X_test]:
    dataset[numerical_features_to_scale] = (
        dataset[numerical_features_to_scale]
        .astype(np.float32)
    )

print("Numerical features converted to float32.")

Numerical features converted to float32.


#### 9.3 Fit the scaler on training data only

In [64]:
# ==========================================================
# 9.3 Fit RobustScaler
# ==========================================================

robust_scaler = RobustScaler()

robust_scaler.fit(
    X_train[numerical_features_to_scale] # We use X_tran, not df_processed, to avoid data leakage
)

print("RobustScaler fitted on the training dataset.")

RobustScaler fitted on the training dataset.


In [65]:
#### 9.4 Apply scaling

In [66]:
# ==========================================================
# 9.4 Transform Train, Validation, and Test Data
# ==========================================================

X_train.loc[:, numerical_features_to_scale] = (
    robust_scaler.transform(
        X_train[numerical_features_to_scale]
    )
)

X_validation.loc[:, numerical_features_to_scale] = (
    robust_scaler.transform(
        X_validation[numerical_features_to_scale]
    )
)

X_test.loc[:, numerical_features_to_scale] = (
    robust_scaler.transform(
        X_test[numerical_features_to_scale]
    )
)

print("Numerical scaling completed successfully.")

Numerical scaling completed successfully.


#### 9.5 Save the scaler and feature list

In [67]:
# ==========================================================
# 9.5 Save Scaling Artifacts
# ==========================================================

PREPROCESSING_PATH = (
    PROJECT_ROOT
    / "models"
    / "preprocessing"
)

PREPROCESSING_PATH.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    robust_scaler,
    PREPROCESSING_PATH / "robust_scaler.pkl"
)

joblib.dump(
    numerical_features_to_scale,
    PREPROCESSING_PATH / "scaled_feature_names.pkl"
)

print("Scaling artifacts saved successfully.")
print(
    f"Scaler path: "
    f"{PREPROCESSING_PATH / 'robust_scaler.pkl'}"
)

Scaling artifacts saved successfully.
Scaler path: /Users/hshazel/Projects/explainable-fraud-investigation-platform/models/preprocessing/robust_scaler.pkl


#### 9.6 Verify the transformation

In [68]:
# ==========================================================
# 9.6 Scaling Verification
# ==========================================================

scaling_verification = pd.DataFrame({
    "Scaled Median": (
        X_train[numerical_features_to_scale]
        .median()
    ),
    "Scaled Q1": (
        X_train[numerical_features_to_scale]
        .quantile(0.25)
    ),
    "Scaled Q3": (
        X_train[numerical_features_to_scale]
        .quantile(0.75)
    )
})

scaling_verification["Scaled IQR"] = (
    scaling_verification["Scaled Q3"]
    - scaling_verification["Scaled Q1"]
)

display(
    scaling_verification
    .head(20)
    .round(3)
)

,Scaled Median,Scaled Q1,Scaled Q3,Scaled IQR
TransactionDT,0.00,-0.52,0.48,1.00
TransactionAmt,0.00,-0.32,0.69,1.00
card1,0.00,-0.45,0.55,1.00
card2,0.00,-0.49,0.51,1.00
card3,0.00,0.00,0.00,0.00
card5,0.00,-1.00,0.00,1.00
addr1,0.00,-0.77,0.23,1.00
addr2,0.00,0.00,0.00,0.00
dist1,0.00,0.00,0.00,0.00
dist2,0.00,0.00,0.00,0.00


#### 9.7 Final integrity checks

In [69]:
# ==========================================================
# 9.7 Scaling Integrity Checks
# ==========================================================

print("=" * 60)
print("Scaling Integrity Checks")
print("=" * 60)

print(
    "Train/Validation columns match:",
    list(X_train.columns)
    == list(X_validation.columns)
)

print(
    "Train/Test columns match:",
    list(X_train.columns)
    == list(X_test.columns)
)

print(
    "Missing values in training:",
    int(X_train.isna().to_numpy().sum())
)

print(
    "Missing values in validation:",
    int(X_validation.isna().to_numpy().sum())
)

print(
    "Missing values in test:",
    int(X_test.isna().to_numpy().sum())
)

print(
    "Infinite values in training:",
    int(
        np.isinf(
            X_train.to_numpy(dtype=np.float32)
        ).sum()
    )
)

print(f"Training shape: {X_train.shape}")
print(f"Validation shape: {X_validation.shape}")
print(f"Test shape: {X_test.shape}")

Scaling Integrity Checks
Train/Validation columns match: True
Train/Test columns match: True
Missing values in training: 0
Missing values in validation: 0
Missing values in test: 0
Infinite values in training: 0
Training shape: (413378, 430)
Validation shape: (88581, 430)
Test shape: (88581, 430)


#### Numerical Feature Scaling Findings

A total of **70 numerical features** were transformed using `RobustScaler`.

One-hot encoded variables and binary indicator variables were excluded from scaling to preserve their original binary representation. The remaining numerical variables were scaled using the median and interquartile range calculated from the training dataset, reducing the influence of outliers and differences in feature magnitude.

The scaler was fitted exclusively on the training data and then applied unchanged to the validation and test datasets, preventing data leakage.

Post-scaling verification confirmed that:
- All datasets retained identical feature sets (430 features).
- No missing values or infinite values were introduced during scaling.
- The median of most scaled numerical variables is approximately 0, while variables with sufficient variability have an interquartile range close to 1, indicating that the `RobustScaler` transformation was applied successfully.
- Several variables retained an interquartile range of 0 because the majority of their observations share the same value after preprocessing. This behavior is expected for low-variance or highly concentrated features and does not indicate an error.

The scaled datasets are now prepared for machine learning models that are sensitive to feature magnitude, including Logistic Regression and neural networks.

### 10. Feature Selection

The objective of this section is to reduce unnecessary model complexity by removing features that contribute little predictive information while preserving variables that may improve fraud detection.

Feature selection is performed using data-driven criteria rather than manual inspection. Constant features are removed because they contain no information, and highly correlated numerical variables are eliminated to reduce redundancy.

Model-based feature importance and explainability techniques (e.g., SHAP) will be applied later during model development to further evaluate feature relevance.

#### 10.1 Remove Constant Features

In [70]:
# ==========================================================
# 10.1 Remove Constant Features
# ==========================================================

constant_features = [
    column
    for column in X_train.columns
    if X_train[column].nunique(dropna=False) <= 1
]

print("=" * 60)
print("Constant Features")
print("=" * 60)

print(f"Constant features found: {len(constant_features)}")

if constant_features:
    display(pd.DataFrame({
        "Feature": constant_features
    }))

Constant Features
Constant features found: 1


,Feature
0,UnknownDevice


#### 10.2 Remove Constant Features from All Sets

In [71]:
# ==========================================================
# 10.2 Remove Constant Features
# ==========================================================

X_train.drop(
    columns=constant_features,
    inplace=True
)

X_validation.drop(
    columns=constant_features,
    inplace=True
)

X_test.drop(
    columns=constant_features,
    inplace=True
)

print(f"Remaining features: {X_train.shape[1]}")

Remaining features: 429


#### 10.3 Correlation Analysis

Only evaluate continuous numerical variables.

In [72]:
# ==========================================================
# 10.3 Correlation Analysis
# ==========================================================

continuous_features = [
    column
    for column in numerical_features_to_scale
    if column in X_train.columns
]

correlation_matrix = (
    X_train[continuous_features]
    .corr()
    .abs()
)

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

correlated_features = [
    column
    for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.95)
]

print("=" * 60)
print("Highly Correlated Features")
print("=" * 60)

print(f"Features to remove: {len(correlated_features)}")

Highly Correlated Features
Features to remove: 109


#### 10.4 Remove Correlated Features

In [73]:
# ==========================================================
# 10.4 Remove Correlated Features
# ==========================================================

X_train.drop(
    columns=correlated_features,
    inplace=True
)

X_validation.drop(
    columns=correlated_features,
    inplace=True
)

X_test.drop(
    columns=correlated_features,
    inplace=True
)

print(f"Remaining features: {X_train.shape[1]}")

Remaining features: 320


#### 10.5 Save Selected Features

In [74]:
# ==========================================================
# 10.5 Save Selected Features
# ==========================================================

selected_features = X_train.columns.tolist()

joblib.dump(
    selected_features,
    PROJECT_ROOT
    / "models"
    / "preprocessing"
    / "selected_features.pkl"
)

print("Selected feature list saved.")

Selected feature list saved.


#### 10.6 Final Dataset Summary

In [75]:
# ==========================================================
# 10.6 Dataset Summary
# ==========================================================

summary = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation",
        "Test"
    ],
    "Rows": [
        len(X_train),
        len(X_validation),
        len(X_test)
    ],
    "Features": [
        X_train.shape[1],
        X_validation.shape[1],
        X_test.shape[1]
    ]
})

display(summary)

,Dataset,Rows,Features
0,Training,413378,320
1,Validation,88581,320
2,Test,88581,320


#### Feature Selection Findings

Feature selection was performed to reduce redundant information while preserving variables with the greatest potential predictive value for fraud detection.

First, constant features containing no variability were identified and removed because they cannot contribute to machine learning models. Next, highly correlated numerical features (absolute Pearson correlation greater than 0.95) were eliminated to reduce multicollinearity, simplify the feature space, and improve computational efficiency while retaining the underlying information.

The same features were removed consistently from the training, validation, and test datasets, ensuring that all datasets maintained identical feature spaces throughout the machine learning pipeline.

As a result, the number of model features was reduced from **430** to **320**, creating a more compact and efficient dataset for model development. More advanced feature selection and explainability techniques, including SHAP values and tree-based feature importance, will be applied during model training and evaluation rather than during preprocessing.

### 11. Save Processed Data

The final step in the preprocessing pipeline is to persist the processed datasets and preprocessing artifacts for reuse in subsequent notebooks.

Saving the processed training, validation, and test datasets ensures that model development can begin immediately without repeating the computationally intensive preprocessing steps. This also improves reproducibility and guarantees that every model is trained using exactly the same feature set.

In addition to the processed datasets, all preprocessing objects—including imputers, encoders, scalers, and the final feature list—have been saved to support future inference and deployment.

#### 11.1 Create Output Directory

In [76]:
# ==========================================================
# 11.1 Create Output Directory
# ==========================================================

PROCESSED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DATA_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Output directory:\n{PROCESSED_DATA_PATH}")

Output directory:
/Users/hshazel/Projects/explainable-fraud-investigation-platform/data/processed


#### 11.2 Save the Datasets

In [77]:
import pyarrow as pa

import pandas as pd

print("PyArrow:", pa.__version__)

print("Pandas:", pd.__version__)

PyArrow: 25.0.0
Pandas: 3.0.3


In [78]:
# ==========================================================
# 11.2 Save Processed Datasets
# ==========================================================
X_train.to_parquet(
    PROCESSED_DATA_PATH / "X_train.parquet",
    index=False
)

X_validation.to_parquet(
    PROCESSED_DATA_PATH / "X_validation.parquet",
    index=False
)

X_test.to_parquet(
    PROCESSED_DATA_PATH / "X_test.parquet",
    index=False
)

y_train.to_frame("isFraud").to_parquet(
    PROCESSED_DATA_PATH / "y_train.parquet",
    index=False
)

y_validation.to_frame("isFraud").to_parquet(
    PROCESSED_DATA_PATH / "y_validation.parquet",
    index=False
)

y_test.to_frame("isFraud").to_parquet(
    PROCESSED_DATA_PATH / "y_test.parquet",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


#### 11.3 Save Transaction IDs

They will be used later for fraud investigations

In [79]:
# ==========================================================
# 11.3 Save Transaction IDs
# ==========================================================

id_train.to_frame("TransactionID").to_parquet(
    PROCESSED_DATA_PATH / "id_train.parquet",
    index=False
)

id_validation.to_frame("TransactionID").to_parquet(
    PROCESSED_DATA_PATH / "id_validation.parquet",
    index=False
)

id_test.to_frame("TransactionID").to_parquet(
    PROCESSED_DATA_PATH / "id_test.parquet",
    index=False
)

print("Transaction IDs saved successfully.")

Transaction IDs saved successfully.


#### 11.4 Summary

In [80]:
# ==========================================================
# 11.4 Saved Files Summary
# ==========================================================

summary = pd.DataFrame({
    "Dataset": [
        "X_train",
        "X_validation",
        "X_test",
        "y_train",
        "y_validation",
        "y_test",
        "id_train",
        "id_validation",
        "id_test"
    ],
    "Rows": [
        len(X_train),
        len(X_validation),
        len(X_test),
        len(y_train),
        len(y_validation),
        len(y_test),
        len(id_train),
        len(id_validation),
        len(id_test)
    ]
})

display(summary)

print()
print(f"Total model features: {X_train.shape[1]}")
print("All processed datasets successfully exported.")

,Dataset,Rows
0,X_train,413378
1,X_validation,88581
2,X_test,88581
3,y_train,413378
4,y_validation,88581
5,y_test,88581
6,id_train,413378
7,id_validation,88581
8,id_test,88581



Total model features: 320
All processed datasets successfully exported.


#### Processed Data Export Findings

The fully preprocessed datasets were successfully exported in Parquet format for use during the model development phase. The training, validation, and test datasets preserve identical feature spaces consisting of **320 engineered features**, ensuring consistency throughout the machine learning pipeline.

The corresponding target variables and transaction identifiers were also saved separately to support model evaluation, fraud investigation, and explainability analyses.

By exporting the processed datasets and preprocessing artifacts, the project achieves full reproducibility and eliminates the need to repeat computationally intensive preprocessing steps during subsequent experimentation.

### 12. Preprocessing Summary and Next Steps

This notebook completed the data preprocessing and feature engineering pipeline for the IEEE-CIS Fraud Detection dataset, transforming the raw transactional data into a clean, structured, and machine learning–ready dataset.

The preprocessing workflow included data quality assessment, missing value treatment, removal of low-information features, feature engineering, categorical encoding, dataset partitioning, numerical feature scaling, and correlation-based feature selection. Each preprocessing step was validated to ensure data integrity and reproducibility.

### Final Dataset Summary

| Metric | Value |
|--------|------:|
| Original observations | 590,540 |
| Training observations | 413,378 |
| Validation observations | 88,581 |
| Test observations | 88,581 |
| Final model features | 320 |
| Missing values | 0 |
| Infinite values | 0 |
| Duplicate rows | 0 |

### Preprocessing Pipeline

- ✓ Data quality assessment
- ✓ Missing value treatment
- ✓ Removal of high-missing features
- ✓ Removal of low-information features
- ✓ Feature engineering
- ✓ Frequency encoding of high-cardinality variables
- ✓ One-hot encoding of low-cardinality variables
- ✓ Stratified train / validation / test split
- ✓ Robust feature scaling
- ✓ Correlation-based feature selection
- ✓ Export of processed datasets and preprocessing artifacts

The processed datasets and preprocessing objects were saved for reuse during model development, ensuring a consistent and reproducible machine learning workflow.

The next notebook will focus on training, tuning, and evaluating multiple machine learning models—including Logistic Regression, Decision Tree, Random Forest, XGBoost, and Multi-Layer Perceptron—to identify fraudulent financial transactions. Model performance will be compared using Accuracy, Precision, Recall, F1-score, ROC-AUC, and Precision-Recall metrics. Explainability techniques, including SHAP, will then be applied to interpret model predictions and support fraud investigation.